In [17]:
#STEP 1: Load libraries
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



import joblib

In [18]:
#STEP 2: Load and Check data
df = pd.read_csv("ctg-studies.csv")

print("Rows and columns:", df.shape)
print("\nStudy Status counts:")
print(df["Study Status"].value_counts())

df.head()

Rows and columns: (61215, 18)

Study Status counts:
Study Status
COMPLETED     46970
TERMINATED     9115
WITHDRAWN      4670
SUSPENDED       460
Name: count, dtype: int64


,NCT Number,Study Status,Brief Summary,Conditions,Interventions,Sponsor,Collaborators,Sex,Age,Phases,Enrollment,Funder Type,Study Type,Study Design,Start Date,Primary Completion Date,Completion Date,Locations
0,NCT05185284,COMPLETED,This is open-labe randomized multicenter compa...,COVID-19,DRUG: Favipiravir|DRUG: Favipiravir|DRUG: Remd...,"Promomed, LLC",Solyur Pharmaceuticals Group,ALL,"ADULT, OLDER_ADULT",PHASE3,217.0,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: P...,2021-08-11,2021-12-30,2021-12-30,"Regional budgetary health care institution ""Iv..."
1,NCT04752033,COMPLETED,This research study is being done to determine...,"Colorectal Surgery|Pain, Postoperative|Analges...",DRUG: Morphine|DRUG: Hydromorphone,Mayo Clinic,NaN,ALL,"ADULT, OLDER_ADULT",PHASE4,80.0,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: S...,2021-03-10,2023-02-17,2023-02-17,"Mayo Clinic in Rochester, Rochester, Minnesota..."
2,NCT06241352,COMPLETED,The results of previous studies conducted by o...,Advanced Pancreatic Cancer,DRUG: statin addition to chemotherapy,Changhai Hospital,NaN,ALL,"ADULT, OLDER_ADULT",PHASE2,42.0,OTHER,INTERVENTIONAL,Allocation: NA | Intervention Model: SINGLE_GR...,2024-03-20,2025-04-30,2025-04-30,"Changhai Hospital, Shanghai, Shanghai Municipa..."
3,NCT03988933,COMPLETED,Rationale:\n\nShorter regimens of high dose da...,Latent Tuberculosis,DRUG: Rifampin double dose|DRUG: Rifampin trip...,McGill University Health Centre/Research Insti...,Canadian Institutes of Health Research (CIHR),ALL,"CHILD, ADULT, OLDER_ADULT",PHASE2,1368.0,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: P...,2019-09-20,2023-01-31,2024-12-31,"Unviversity of Calgary, Calgary, Alberta, Cana..."
4,NCT06149247,COMPLETED,The objective of this clinical study is to com...,Cutaneous T-Cell Lymphoma/Mycosis Fungoides,DRUG: Hypericin|DRUG: Mechlorethamine Topical Gel,Soligenix,NaN,ALL,"ADULT, OLDER_ADULT",PHASE2,10.0,INDUSTRY,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: P...,2023-12-05,2024-05-31,2024-06-27,"Rochester Skin Lymphoma Medical Group, Fairpor..."


In [19]:
#STEP 3: Create target and duration features
df["target_status"] = df["Study Status"].map({
    "COMPLETED": 1,
    "TERMINATED": 0,
    "WITHDRAWN": 0,
    "SUSPENDED": 0
})

df["Start Date"] = pd.to_datetime(df["Start Date"], errors="coerce", format="mixed")
df["Completion Date"] = pd.to_datetime(df["Completion Date"], errors="coerce", format="mixed")
df["Primary Completion Date"] = pd.to_datetime(df["Primary Completion Date"], errors="coerce", format="mixed")

df["start_year"] = df["Start Date"].dt.year

df["duration_months"] = (
    (df["Completion Date"] - df["Start Date"]).dt.days / 30.44
)

print("Missing duration values:", df["duration_months"].isna().sum())
print("Negative duration values:", (df["duration_months"] < 0).sum())

df[["Study Status", "target_status", "Start Date", "Completion Date", "start_year", "duration_months"]].head()

Missing duration values: 149
Negative duration values: 0


,Study Status,target_status,Start Date,Completion Date,start_year,duration_months
0,COMPLETED,1,2021-08-11,2021-12-30,2021,4.632063
1,COMPLETED,1,2021-03-10,2023-02-17,2021,23.291721
2,COMPLETED,1,2024-03-20,2025-04-30,2024,13.337714
3,COMPLETED,1,2019-09-20,2024-12-31,2019,63.370565
4,COMPLETED,1,2023-12-05,2024-06-27,2023,6.734560


In [20]:
#STEP 4: Keeping only completed trials with valid duration

#Duration model should learn full study duration only from completed trials
duration_df = df[
    (df["target_status"] == 1) &
    (df["duration_months"].notna()) &
    (df["duration_months"] > 0)
].copy()

print("Original data shape:", df.shape)
print("Duration modelling data shape:", duration_df.shape)

print("\nDuration summary:")
print(duration_df["duration_months"].describe())

Original data shape: (61215, 21)
Duration modelling data shape: (46845, 21)

Duration summary:
count    46845.000000
mean        25.395414
std         21.991336
min          0.032852
25%          8.574244
50%         18.955322
75%         36.103811
max        133.869908
Name: duration_months, dtype: float64


In [21]:
#STEP 5: Create predictors(X) and target(y) for regression
remove_cols = [
    "NCT Number",
    "Study Status",
    "Start Date",
    "Primary Completion Date",
    "Completion Date",
    "duration_months",
    "target_status",
    "Brief Summary",
    "Collaborators",
    "Study Type"
]

X = duration_df.drop(columns=remove_cols, errors="ignore")
y = duration_df["duration_months"]

print("Predictor columns:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Predictor columns:
['Conditions', 'Interventions', 'Sponsor', 'Sex', 'Age', 'Phases', 'Enrollment', 'Funder Type', 'Study Design', 'Locations', 'start_year']

X shape: (46845, 11)
y shape: (46845,)


In [22]:
# STEP 6: Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining duration summary:")
print(y_train.describe())

print("\nTesting duration summary:")
print(y_test.describe())

Training shape: (37476, 11)
Testing shape: (9369, 11)

Training duration summary:
count    37476.000000
mean        25.350518
std         21.983230
min          0.032852
25%          8.574244
50%         18.955322
75%         36.005256
max        133.869908
Name: duration_months, dtype: float64

Testing duration summary:
count    9369.000000
mean       25.574998
std        22.023991
min         0.032852
25%         8.574244
50%        19.053876
75%        36.530880
max       132.457293
Name: duration_months, dtype: float64


In [23]:
#STEP 7: Feature engineering 
class FeatureBuilder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        sponsor = X["Sponsor"].fillna("Unknown")
        self.sponsor_counts_ = sponsor.value_counts()
        return self

    def transform(self, X):
        X = X.copy()
        
        # Fill only the fields needed for feature creation
        for col in ["Sex", "Funder Type", "Study Design"]:
            X[col] = X[col].fillna("Unknown")

        X["Conditions"] = X["Conditions"].fillna("")
        X["Interventions"] = X["Interventions"].fillna("")

        # Count features
        X["condition_count"] = X["Conditions"].apply(
            lambda x: len([v for v in str(x).split("|") if v.strip()])
        )

        X["intervention_count"] = X["Interventions"].apply(
            lambda x: len([v for v in str(x).split("|") if v.strip()])
        )
        
        # Location features
        loc = X["Locations"]
        loc_text = loc.fillna("").str.lower()

        X["has_location"] = loc.notna().astype(int)

        X["location_count"] = loc.fillna("").apply(
            lambda x: len([v for v in str(x).split("|") if v.strip()])
        )

        X["is_multisite"] = (X["location_count"] > 1).astype(int)

        X["has_us_location"] = loc_text.str.contains("united states", regex=False).astype(int)
        X["has_uk_location"] = loc_text.str.contains("united kingdom", regex=False).astype(int)
        X["has_china_location"] = loc_text.str.contains("china", regex=False).astype(int)
        X["has_canada_location"] = loc_text.str.contains("canada", regex=False).astype(int)

        # Age flags using exact matching
        def has_age_group(value, group):
            values = [v.strip() for v in str(value).upper().split(",")]
            return int(group in values)

        X["has_child"] = X["Age"].apply(lambda x: has_age_group(x, "CHILD"))
        X["has_adult"] = X["Age"].apply(lambda x: has_age_group(x, "ADULT"))
        X["has_older_adult"] = X["Age"].apply(lambda x: has_age_group(x, "OLDER_ADULT"))

        # Phase flags using exact matching
        def has_phase(value, phase):
            values = [v.strip() for v in str(value).upper().split("|")]
            return int(phase in values)

        X["has_early_phase1"] = X["Phases"].apply(lambda x: has_phase(x, "EARLY_PHASE1"))
        X["has_phase1"] = X["Phases"].apply(lambda x: has_phase(x, "PHASE1"))
        X["has_phase2"] = X["Phases"].apply(lambda x: has_phase(x, "PHASE2"))
        X["has_phase3"] = X["Phases"].apply(lambda x: has_phase(x, "PHASE3"))
        X["has_phase4"] = X["Phases"].apply(lambda x: has_phase(x, "PHASE4"))

        # Intervention type flags
        intervention = X["Interventions"].fillna("").str.upper()

        X["has_drug"] = intervention.str.contains("DRUG:", regex=False).astype(int)
        X["has_device"] = intervention.str.contains("DEVICE:", regex=False).astype(int)
        X["has_biological"] = intervention.str.contains("BIOLOGICAL:", regex=False).astype(int)
        X["has_procedure"] = intervention.str.contains("PROCEDURE:", regex=False).astype(int)
        X["has_behavioral"] = intervention.str.contains("BEHAVIORAL:", regex=False).astype(int)

         # Covers both possible spellings
        X["has_diagnostic_test"] = (
            intervention.str.contains("DIAGNOSTIC_TEST:", regex=False) |
            intervention.str.contains("DIAGNOSTIC TEST:", regex=False)
        ).astype(int)

        # Sponsor frequency learned from training data
        sponsor = X["Sponsor"].fillna("Unknown")
        X["sponsor_frequency"] = sponsor.map(self.sponsor_counts_).fillna(1)

        # Extract useful parts from Study Design
        def get_design_value(text, key):
            for part in str(text).split("|"):
                part = part.strip()
                if part.upper().startswith(key.upper() + ":"):
                    return part.split(":", 1)[1].strip()
            return "Unknown"

        X["allocation"] = X["Study Design"].apply(lambda x: get_design_value(x, "Allocation"))
        X["intervention_model"] = X["Study Design"].apply(lambda x: get_design_value(x, "Intervention Model"))
        X["masking"] = X["Study Design"].apply(lambda x: get_design_value(x, "Masking"))
        X["primary_purpose"] = X["Study Design"].apply(lambda x: get_design_value(x, "Primary Purpose"))

        return X

builder = FeatureBuilder()

X_train_fe = builder.fit_transform(X_train)
X_test_fe = builder.transform(X_test)

print("Before feature engineering:", X_train.shape, X_test.shape)
print("After feature engineering:", X_train_fe.shape, X_test_fe.shape)

display(X_train_fe[[
    "Age", "has_child", "has_adult", "has_older_adult",
    "Phases", "has_phase1", "has_phase2", "has_phase3", "has_phase4"
]].head())

Before feature engineering: (37476, 11) (9369, 11)
After feature engineering: (37476, 39) (9369, 39)


,Age,has_child,has_adult,has_older_adult,Phases,has_phase1,has_phase2,has_phase3,has_phase4
19064,ADULT,0,1,0,PHASE1,1,0,0,0
19342,"ADULT, OLDER_ADULT",0,1,1,PHASE3,0,0,1,0
27340,"ADULT, OLDER_ADULT",0,1,1,PHASE2,0,1,0,0
40620,ADULT,0,1,0,PHASE3,0,0,1,0
31116,"CHILD, ADULT, OLDER_ADULT",1,1,1,PHASE3,0,0,1,0


In [24]:
#STEP 8: Droping replaced raw columns and preparing final feature groups
raw_cols_to_drop = [
    "Age",
    "Phases",
    "Sponsor",
    "Locations",
    "Study Design"
]

X_train_model = X_train_fe.drop(columns=raw_cols_to_drop, errors="ignore")
X_test_model = X_test_fe.drop(columns=raw_cols_to_drop, errors="ignore")

print("X_train_model:", X_train_model.shape)
print("X_test_model:", X_test_model.shape)

print("\nMissing values:")
print(X_train_model.isnull().sum().sort_values(ascending=False).head(10))

X_train_model: (37476, 34)
X_test_model: (9369, 34)

Missing values:
Enrollment        5
Conditions        0
has_procedure     0
has_phase2        0
has_phase3        0
has_phase4        0
has_drug          0
has_device        0
has_biological    0
has_behavioral    0
dtype: int64


In [25]:
# STEP 9: Final Preprocessing - Imputation, scaling, OHE, TF-IDF for Linear Regression and XGBoost Regression

numeric_cols = [
    "Enrollment",
    "start_year",
    "condition_count",
    "intervention_count",
    "location_count",
    "sponsor_frequency"
]

binary_cols = [
    "has_location",
    "is_multisite",
    "has_us_location",
    "has_uk_location",
    "has_china_location",
    "has_canada_location",
    "has_child",
    "has_adult",
    "has_older_adult",
    "has_early_phase1",
    "has_phase1",
    "has_phase2",
    "has_phase3",
    "has_phase4",
    "has_drug",
    "has_device",
    "has_biological",
    "has_procedure",
    "has_behavioral",
    "has_diagnostic_test"
]

categorical_cols = [
    "Sex",
    "Funder Type",
    "allocation",
    "intervention_model",
    "masking",
    "primary_purpose"
]

# Limit the number of TF-IDF text features so Conditions and Interventions do not create thousands of columns.
max_text_features = 200


# Linear Regression is sensitive to feature scale.
# Therefore, numeric columns are imputed and then scaled.
numeric_pipe_linear = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# XGBoost is tree-based and does not need numeric scaling.
# Therefore, numeric columns are only imputed.
numeric_pipe_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])


# Categorical columns:
# Missing values are filled with "Unknown", then One-Hot Encoding converts them into 0/1 columns.
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# Preprocessor for Linear Regression:
# Uses scaled numeric features.
preprocessor_linear = ColumnTransformer([
    ("numeric", numeric_pipe_linear, numeric_cols),
    ("binary", "passthrough", binary_cols),
    ("categorical", categorical_pipe, categorical_cols),
    ("conditions_text", TfidfVectorizer(max_features=max_text_features), "Conditions"),
    ("interventions_text", TfidfVectorizer(max_features=max_text_features), "Interventions")
])


# Preprocessor for XGBoost Regression:
# Uses unscaled numeric features.
preprocessor_xgb = ColumnTransformer([
    ("numeric", numeric_pipe_xgb, numeric_cols),
    ("binary", "passthrough", binary_cols),
    ("categorical", categorical_pipe, categorical_cols),
    ("conditions_text", TfidfVectorizer(max_features=max_text_features), "Conditions"),
    ("interventions_text", TfidfVectorizer(max_features=max_text_features), "Interventions")
])


# Fit preprocessing on training data only.
X_train_linear = preprocessor_linear.fit_transform(X_train_model)

# Apply the same learned preprocessing rules to the test set.
X_test_linear = preprocessor_linear.transform(X_test_model)


# Repeat the same process for the XGBoost-specific preprocessor.
X_train_xgb = preprocessor_xgb.fit_transform(X_train_model)
X_test_xgb = preprocessor_xgb.transform(X_test_model)


print("Linear Regression data:", X_train_linear.shape, X_test_linear.shape)
print("XGBoost Regression data:", X_train_xgb.shape, X_test_xgb.shape)

Linear Regression data: (37476, 477) (9369, 477)
XGBoost Regression data: (37476, 477) (9369, 477)


In [26]:
# Step 10:  Train baseline regression model
# Linear Regression is used as a simple baseline regression model.


linear_model = LinearRegression()

# Use the Linear Regression preprocessed data
linear_model.fit(X_train_linear, y_train)

linear_pred = linear_model.predict(X_test_linear)

print("Linear Regression trained.")

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression Results")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R²:", linear_r2)

Linear Regression trained.
Linear Regression Results
MAE: 11.98171042454145
RMSE: 16.11533484422272
R²: 0.46453265387108467


In [27]:
# Step 11: Train XGBoost regression model
xgb_reg_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# Use the XGBoost-specific preprocessed data
xgb_reg_model.fit(X_train_xgb, y_train)

xgb_pred = xgb_reg_model.predict(X_test_xgb)

print("XGBoost regression model trained.")

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost Regression Results")
print("MAE:", xgb_mae)
print("RMSE:", xgb_rmse)
print("R²:", xgb_r2)

XGBoost regression model trained.
XGBoost Regression Results
MAE: 11.132471833484262
RMSE: 15.184855404760942
R²: 0.5245819723481339


In [28]:
# Step 12 — Compare regression model performance

regression_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "XGBoost Regressor"],
    "MAE": [linear_mae, xgb_mae],
    "RMSE": [linear_rmse, xgb_rmse],
    "R²": [linear_r2, xgb_r2]
})

regression_comparison.round(3)

,Model,MAE,RMSE,R²
0,Linear Regression,11.982,16.115,0.465
1,XGBoost Regressor,11.132,15.185,0.525


In [29]:
# Step 13 — Select final regression model

final_regression_model_name = "XGBoost Regressor"
final_regression_model = xgb_reg_model

print("Final selected regression model:", final_regression_model_name)

Final selected regression model: XGBoost Regressor


In [30]:
# Step 14 — Create prediction results table for XGBoost Regression

duration_results = pd.DataFrame({
    "actual_duration_months": y_test.values,
    "predicted_duration_months": xgb_pred
})

duration_results["error_months"] = (
    duration_results["actual_duration_months"] -
    duration_results["predicted_duration_months"]
).abs()

# Show a simple sample of prediction examples
duration_results.head(10).round(2)

,actual_duration_months,predicted_duration_months,error_months
0,30.95,26.950001,4.00
1,2.00,15.420000,13.41
2,97.01,38.029999,58.98
3,78.68,63.279999,15.40
4,8.87,38.240002,29.37
5,2.69,18.080000,15.39
6,37.52,35.770000,1.75
7,28.81,22.400000,6.41
8,3.25,2.140000,1.11
9,22.96,18.730000,4.24


In [31]:
# Step 15 — Save final duration prediction model files

import joblib

joblib.dump(builder, "duration_feature_builder.pkl")
joblib.dump(raw_cols_to_drop, "duration_raw_cols_to_drop.pkl")
joblib.dump(preprocessor_xgb, "duration_preprocessor.pkl")
joblib.dump(xgb_reg_model, "duration_xgboost_model.pkl")

print("Saved files:")
print("duration_feature_builder.pkl")
print("duration_raw_cols_to_drop.pkl")
print("duration_preprocessor.pkl")
print("duration_xgboost_model.pkl")

Saved files:
duration_feature_builder.pkl
duration_raw_cols_to_drop.pkl
duration_preprocessor.pkl
duration_xgboost_model.pkl
